# Chapitre 9 - Cours 5 : le consensus en pratique, Raft

Le cours précédent a montré que le consensus équivaut à un **journal partagé**, et
qu'un journal tenu par un leader unique est facile... tant que le leader ne tombe pas.
Les algorithmes de consensus utilisés en pratique (Raft, Multi-Paxos, Zab,
Viewstamped Replication) sont essentiellement de la **réplication à leader unique avec
élection et bascule automatiques, sans cerveau divisé ni perte d'écriture validée**. Ce
notebook implémente **Raft** sur le simulateur (`cs.RaftNode`), l'observe sur des
diagrammes espace-temps, le soumet à des pannes et à des partitions, vérifie sa sûreté
et mesure ce qu'il coûte. Il couvre la fin de la section *Fault-Tolerant Consensus* : *Consensus algorithms and total order broadcast*, *Single-leader replication and consensus*, *Epoch numbering and quorums*, *Limitations of consensus*.

**Plan.**

1. De la réplication à leader unique au consensus.
2. Mandats et quorums qui se recoupent.
3. Raft sur le simulateur : l'élection.
4. Réplication du journal et validation par majorité.
5. Pannes et partitions : le leader isolé, les lectures linéarisables.
6. Le nouveau leader doit être à jour.
7. Avantages et limites du consensus.

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ddia import consistency as cs
from ddia.diagrams import Diagram
from ddia.plotting import (AMBER, COLORS, GRAY, GREEN, INDIGO, LIGHT, ROSE, SKY, SLATE, TEAL, VIOLET,
                           annotate_bars, set_style, subtitle)
from ddia.timelines import SpaceTime

set_style()
plt.rcParams["axes.titlepad"] = 16
pd.set_option("display.precision", 3)
TERM_COLORS = [GRAY, INDIGO, TEAL, AMBER, VIOLET, SKY, ROSE, GREEN, SLATE]


def dessine_journaux(nodes, ax, titre, sous_titre=""):
    """Grille des journaux : une ligne par nœud, une case par entrée colorée par mandat ;
    trait vert = indice de validation connu du nœud, contour épais = leader."""
    names = list(nodes)
    width = max(len(n.log) for n in nodes.values())
    for i, x in enumerate(names):
        nd = nodes[x]
        y = len(names) - 1 - i
        for k, e in enumerate(nd.log[1:], start=1):
            c = TERM_COLORS[e.term % len(TERM_COLORS)]
            cmd = e.cmd
            lab = "·" if cmd[0] == "noop" else (f"x={cmd[2]}" if cmd[0] == "set" else
                                                 ("lect." if cmd[0] == "read" else f"cas→{cmd[3]}"))
            ax.add_patch(plt.Rectangle((k - 0.45, y - 0.35), 0.9, 0.7, facecolor=LIGHT.get(c, "#F1F5F9"),
                                       edgecolor=c, lw=1.4))
            ax.text(k, y + 0.08, lab, ha="center", va="center", fontsize=8, color=SLATE)
            ax.text(k, y - 0.22, f"m{e.term}", ha="center", va="center", fontsize=6.5, color=c)
        if nd.commit:
            ax.plot([nd.commit + 0.5] * 2, [y - 0.45, y + 0.45], color=GREEN, lw=2.5)
        state = "leader" if nd.role == "leader" else ("en panne" if not nd.up else nd.role.replace("follower", "suiveur").replace("candidate", "candidat"))
        ax.text(-0.2, y, f"{x} ({state}, m{nd.term})", ha="right", va="center", fontsize=8.5,
                color=ROSE if not nd.up else SLATE, fontweight="bold" if nd.role == "leader" else "normal")
    ax.set_xlim(-0.1, width + 0.2); ax.set_ylim(-0.6, len(names) - 0.4)
    ax.set_xticks(range(1, width)); ax.set_yticks([]); ax.grid(False)
    for s_ in ("left", "top", "right"):
        ax.spines[s_].set_visible(False)
    ax.set_xlabel("indice dans le journal")
    ax.set_title(titre)
    if sous_titre:
        subtitle(ax, sous_titre)


def zoom(st, t0, t1):
    """Restreint un diagramme espace-temps à [t0, t1] en gardant les noms des nœuds visibles."""
    st.ax.set_xlim(t0 - 0.02 * (t1 - t0), t1)
    for txt in st.ax.texts[:len(st.nodes)]:
        txt.set_x(t0 - 0.015 * (t1 - t0))


def leader_actuel(nodes):
    ls = [x for x, n in nodes.items() if n.role == "leader" and n.up]
    return max(ls, key=lambda x: nodes[x].term) if ls else None